# FRED Data Acquisition
Pulling risk-free rate, CPI, VIX, 10-year yield, Fed funds rate, unemployment rate, and NBER recession indicator from FRED.

In [4]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from fredapi import Fred

## Setup paths and API connection

In [2]:
RAW_DATA_PATH = Path('../data/raw')
PROCESSED_DATA_PATH = Path('../data/processed')

RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
load_dotenv()  # reads FRED_API_KEY from .env
FRED_API_KEY = os.getenv('FRED_API_KEY')

assert FRED_API_KEY is not None, "FRED_API_KEY not found - check your .env file"

fred = Fred(api_key=FRED_API_KEY)
print("Connected to FRED API")

Connected to FRED API


## Series to pull

| Series ID | Description | Frequency |
|---|---|---|
| DGS3MO | 3-Month Treasury Yield (risk-free rate) | Daily |
| CPIAUCSL | Consumer Price Index | Monthly |
| VIXCLS | CBOE Volatility Index (VIX) | Daily |
| DGS10 | 10-Year Treasury Yield | Daily |
| FEDFUNDS | Federal Funds Rate | Monthly |
| UNRATE | Unemployment Rate | Monthly |
| USREC | NBER Recession Indicator (1=recession) | Monthly |`

In [5]:
series_map = {
    'DGS3MO': 'risk_free_rate_pct',
    'CPIAUCSL': 'cpi_index',
    'VIXCLS': 'vix',
    'DGS10': 'treasury_10yr_pct',
    'FEDFUNDS': 'fed_funds_rate_pct',
    'UNRATE': 'unemployment_rate_pct',
    'USREC': 'recession_flag',
}

raw_series = {}
for series_id, col_name in series_map.items():
    data = fred.get_series(series_id)
    df = data.to_frame(name=col_name)
    df.index.name = 'date'
    raw_series[series_id] = df
    print(f"{series_id} -> {col_name}: {len(df)} rows, {df.index.min()} to {df.index.max()}")

DGS3MO -> risk_free_rate_pct: 11700 rows, 1981-09-01 00:00:00 to 2026-07-06 00:00:00
CPIAUCSL -> cpi_index: 953 rows, 1947-01-01 00:00:00 to 2026-05-01 00:00:00
VIXCLS -> vix: 9526 rows, 1990-01-02 00:00:00 to 2026-07-07 00:00:00
DGS10 -> treasury_10yr_pct: 16830 rows, 1962-01-02 00:00:00 to 2026-07-06 00:00:00
FEDFUNDS -> fed_funds_rate_pct: 864 rows, 1954-07-01 00:00:00 to 2026-06-01 00:00:00
UNRATE -> unemployment_rate_pct: 942 rows, 1948-01-01 00:00:00 to 2026-06-01 00:00:00
USREC -> recession_flag: 2059 rows, 1854-12-01 00:00:00 to 2026-06-01 00:00:00


## Quick sanity check on one series

In [6]:
raw_series['VIXCLS'].head()

,vix
date,
1990-01-02,17.24
1990-01-03,18.19
1990-01-04,19.22
1990-01-05,20.11
1990-01-08,20.26


## Save raw data
Each series saved individually - preprocessing notebook handles merging and alignment.

In [8]:
for series_id, col_name in series_map.items():
    filename = f"fred_{col_name}_raw.csv"
    raw_series[series_id].to_csv(RAW_DATA_PATH / filename)
    print(f"Saved {filename}")

Saved fred_risk_free_rate_pct_raw.csv
Saved fred_cpi_index_raw.csv
Saved fred_vix_raw.csv
Saved fred_treasury_10yr_pct_raw.csv
Saved fred_fed_funds_rate_pct_raw.csv
Saved fred_unemployment_rate_pct_raw.csv
Saved fred_recession_flag_raw.csv
